# ステージ2: リファレンス構築

eptA oligo metatranscriptome（`261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome`）

ゲート1を通過したので、マッピング用のリファレンスを作る。**このnotebookの成果物が
ステージ3以降のすべての前提になる。**

## 作るもの

```
ref/oligo_eptA.fna      統合FASTA（4株 + pME008 + mouse decoy）
ref/oligo_eptA.gtf      統合GTF（株ID・CDSフラグ・KO・eptA3分割・mappability）
ref/bt2/                bowtie2 index
ref/salmon/             salmon index
```

## ゲート1で確定済みの前提

| 項目 | 決定 |
|---|---|
| `Odo_St24` | **リファレンスに入れない。** St23 とゲノムがほぼ同一（1kb以上の固有配列 0 bp）。カウントは「*Odoribacter* sp.（CE91-St23/St24）」という1単位 |
| `Bth_VPI` ↔ `Bth_St8` | **株レベルで区別可能**（固有 199 kb / 206 CDS、両株とも diamond blastp で検証済み）。共通領域がどこまで使えるかは §J の mappability で確定する |
| `pME008` | **両群に存在する**（W群=エンプティー、K群=eptA標的）。入れないと両群とも unmapped |
| mouse | **decoy として入れる。** 宿主RNAの誤マップを防ぐ |

## 進め方

A〜C は入力の確認と eptA 座標の確定。D〜F は重い処理（Bakta / eggNOG）。
G〜I でリファレンスを組み、J で mappability を測る。

**§C で VPI と B.uni のゲノム座標が確定する。** これが GTF の eptA feature に必要。

## セル1: 設定
**ここだけ環境に合わせて書き換える。**

In [ ]:
from pathlib import Path

# ── 中間ファイル（容量大）。ステージ1と同じ scratch
ROOT    = Path("/mnt/scratch/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome")
# ── notebook と小さい表（git管理側）
PROJECT = Path("/home/tanoue/261016_Xiaoxi_eptAoligo_2wFeces_Metatranscriptome")

# ── 入力 ─────────────────────────────────────────────────────
GENOME_DIR = Path("/mnt/scratch/tanoue/genome/261016_Xiaoxi_eptAoligo_5strains/fna")

# eptA 変異部位のファイル群（Mac から転送したもの）
#   BthetaVPI_eptA_WT_site/ などの親ディレクトリを指す
EPTA_DIR   = ROOT / "eptA_mutant_sites"

# pME008 の GenBank（EPTA_DIR 配下にあるのでそこから拾う。無ければ直接指定）
PME008_GB  = None          # 例: EPTA_DIR/"OdoribacteriaceaCE91_st23_eptA_WT_site/plasmid/pME008.gb"

# マウスゲノム（decoy）。primary assembly の FASTA
MOUSE_FNA  = Path("/mnt/hdd2/Genome/mouse/GRCm39.primary_assembly.genome.fa")   # ★要確認

# ── 解析に使う株（Odo_St24 は入れない。ゲート1の STOP 判定） ──
STRAINS = ["Bth_VPI", "Bth_St8", "Bun_8492", "Odo_St23"]
EXCLUDED = {"Odo_St24": "ゲート1 STOP: Odo_St23 とゲノムがほぼ同一（1kb以上の固有配列 0 bp）"}

READLEN = 150          # 150 bp PE
THREADS = 16

# ── conda 環境（notebook から conda activate は効かないので絶対パスで叩く）──
CONDA   = Path("/home/tanoue/miniforge3")
METATX  = CONDA/"envs/metatx/bin"
CHECKM2 = CONDA/"envs/checkm2/bin"
EMAPPER = CONDA/"envs/eggnog-mapper/bin"

# ── Bakta の DB（§E の冒頭で方針を決める）────────────────────
BAKTA_DB = Path("/mnt/hdd2/db/bakta")          # ★未ダウンロードなら §E を読むこと
EGGNOG_DB = Path("/mnt/hdd2/db/eggnog")        # ★

# ── 出力 ─────────────────────────────────────────────────────
REF_DIR  = ROOT/"ref"; BAKTA_DIR = ROOT/"bakta"; EMAP_DIR = ROOT/"eggnog"
QC_DIR   = ROOT/"refqc"; MAP_DIR = ROOT/"mappability"; LOG_DIR = ROOT/"logs"
for d in (REF_DIR, BAKTA_DIR, EMAP_DIR, QC_DIR, MAP_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("ROOT     :", ROOT)
print("STRAINS  :", STRAINS, " 除外:", list(EXCLUDED))
print("READLEN  :", READLEN, "bp / THREADS:", THREADS)

## セル2: ヘルパー
ステージ1と同じ。conda 環境の bin を順に探して絶対パスを返す。

In [ ]:
import subprocess, shutil, time, textwrap, re, struct, os, gzip, csv
from collections import defaultdict, Counter
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200); pd.set_option("display.max_rows", 300)

def sh(cmd, log=None, quiet=False, timeout=None):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if log: Path(log).write_text(r.stderr)
    if r.returncode != 0:
        print(f"[失敗 rc={r.returncode}] {cmd[:120]}")
        print(textwrap.indent(r.stderr[-2000:], "  "))
        return None
    if not quiet: print(f"[完了 {time.time()-t0:.1f}s] {cmd.split()[0]}")
    return r.stdout

TOOL_DIRS = [METATX, CHECKM2, EMAPPER, CONDA/"bin"]
def tool_path(name):
    for d in TOOL_DIRS:
        p = Path(d)/name
        if p.exists(): return str(p)
    return shutil.which(name)
def have(name): return tool_path(name) is not None

def read_fasta(path):
    op = gzip.open if str(path).endswith(".gz") else open
    name, seq = None, []
    with op(path, "rt") as f:
        for line in f:
            if line.startswith(">"):
                if name: yield name, "".join(seq)
                name, seq = line[1:].strip(), []
            else: seq.append(line.strip())
    if name: yield name, "".join(seq)

def write_fasta(path, records, width=60):
    with open(path, "w") as f:
        for n, s in records:
            f.write(f">{n}\n")
            for i in range(0, len(s), width): f.write(s[i:i+width]+"\n")

def show(df, title=None):
    if title: print(title)
    try:
        from IPython.display import display; display(df)
    except ImportError:
        print(df.to_string(index=False))

print("ヘルパー定義完了")

---
# A. 入力の確認

ゲノム4本、eptA 変異部位のファイル群、pME008、マウスゲノムが揃っているか。

## セル3: ゲノムと周辺ファイルの確認

In [ ]:
genomes = {}
for ext in ("*.fna", "*.fasta", "*.fa"):
    for f in sorted(GENOME_DIR.glob(ext)):
        if f.stem in STRAINS: genomes[f.stem] = f

rows = []
for sid in STRAINS:
    f = genomes.get(sid)
    if f is None:
        rows.append({"株ID": sid, "状態": "!! ファイルなし"}); continue
    lens = [len(s) for _, s in read_fasta(f)]
    rows.append({"株ID": sid, "contig数": len(lens), "全長_Mb": round(sum(lens)/1e6, 3),
                 "最長_Mb": round(max(lens)/1e6, 3), "状態": "OK"})
show(pd.DataFrame(rows), "解析に使うゲノム")

print()
for sid, why in EXCLUDED.items():
    print(f"除外: {sid}  — {why}")

print()
print("eptA 変異部位フォルダ:", EPTA_DIR, "存在:", EPTA_DIR.exists())
if EPTA_DIR.exists():
    n = sum(1 for _ in EPTA_DIR.rglob("*") if _.is_file())
    print(f"  ファイル {n} 件")
else:
    print("  !! Mac の 'eptA mutant sites' フォルダをサーバへ転送してください")

print()
print("マウスゲノム:", MOUSE_FNA, "存在:", MOUSE_FNA.exists())
if not MOUSE_FNA.exists():
    print("  !! decoy に使うので必要。既存のものを探すか、GENCODE/Ensembl から取得する")
    print("     例: ls /mnt/hdd2/Genome/ | grep -i -E 'mouse|GRCm|mm39|mm10'")

## セル4: pME008 の GenBank を探す

両群に存在するので**必ずリファレンスに入れる**。15,808 bp、Type I-F CAST
（Cas6/7/8 + TniQ + TnsA/B/C）+ tetX1 + PcfxA + E. coli ori。

In [ ]:
if PME008_GB is None:
    cand = sorted(EPTA_DIR.rglob("pME008*.gb")) + sorted(EPTA_DIR.rglob("pME008*.gbk"))
    PME008_GB = cand[0] if cand else None

print("pME008 GenBank:", PME008_GB)
if PME008_GB and Path(PME008_GB).exists():
    txt = Path(PME008_GB).read_text(errors="replace")
    m = re.search(r"LOCUS\s+(\S+)\s+(\d+)\s+bp", txt)
    print(f"  LOCUS: {m.group(1)}  長さ: {int(m.group(2)):,} bp" if m else "  LOCUS行が読めません")
    labels = sorted(set(re.findall(r'/label="([^"]+)"', txt)))
    key = [l for l in labels if re.search(r"Cas|Tns|TniQ|tetX|PcfxA|Ori|Transposon|flank", l, re.I)]
    print("  主要feature:", ", ".join(key[:14]))
else:
    print("  !! 見つかりません。EPTA_DIR を確認するか PME008_GB を直接指定してください")

---
# B. eptA 変異座標の解析

**Mac 上で行った解析を、サーバ上で再現できる形にしたもの。**
WT と変異体の配列を突き合わせて、欠失・挿入の区間を自動で求める。

## 仕組み

WT と変異体は「共通の5'端 + 可変部 + 共通の3'端」という構造をしている。
5'端と3'端の一致長を数えれば、可変部（= 変異区間）が決まる。

```
WT  : [-------共通5'端-------][--欠失される配列--][--共通3'端--]
MUT : [-------共通5'端-------]                   [--共通3'端--]
```

挿入の場合は逆に、変異体側の可変部が挿入配列になる。

## ファイル形式の注意

変異部位のファイルは3種類が混在している。

| 拡張子 | 実際の中身 | 対象 |
|---|---|---|
| `.gb` | GenBank テキスト | Bth_St8, Odo_St23, Odo_St24 |
| `.dna` | **SnapGene バイナリ** | Bth_VPI / Bun_8492 の **WT** |
| `.dna` | **GenBank テキスト**（拡張子が紛らわしい） | Bth_VPI / Bun_8492 の **変異体** |

先頭8バイトが `LOCUS` かどうかで判別する。

## セル5: 配列・feature 読み取り関数

In [ ]:
def _sg_blocks(path):
    """SnapGene .dna は [1バイトtype][4バイトBE長][データ] の連結"""
    d = open(path, "rb").read(); i = 0
    while i + 5 <= len(d):
        t = d[i]; n = struct.unpack(">I", d[i+1:i+5])[0]; i += 5
        yield t, d[i:i+n]; i += n

def _gb_seq(path):
    s, on = [], False
    for line in open(path, encoding="utf-8", errors="replace"):
        if line.startswith("ORIGIN"): on = True; continue
        if on:
            if line.startswith("//"): break
            s.append(re.sub(r"[^acgtnACGTN]", "", line))
    return "".join(s).upper()

def _sg_seq(path):
    out = None
    for t, b in _sg_blocks(path):
        if t == 0: out = b[1:].decode("ascii", "replace").upper()
    return out

def load_seq(path):
    """.gb / SnapGene .dna / GenBankなのに.dna の3種すべてに対応"""
    return _gb_seq(path) if open(path, "rb").read(8).startswith(b"LOCUS") else _sg_seq(path)

def load_features(path):
    """(name, type, start, end) のリスト。座標は1-based inclusive"""
    feats = []
    if open(path, "rb").read(8).startswith(b"LOCUS"):
        # GenBank: FEATURES 〜 ORIGIN を読む
        blk, cur = [], None
        for line in open(path, encoding="utf-8", errors="replace"):
            if line.startswith("ORIGIN"): break
            m = re.match(r"^ {5}(\S+)\s+(\S.*)$", line)
            if m:
                if cur: blk.append(cur)
                cur = {"type": m.group(1), "loc": m.group(2).strip(), "label": None}
            elif cur and "/label=" in line:
                lm = re.search(r'/label="?([^"\n]+)"?', line)
                if lm and cur["label"] is None: cur["label"] = lm.group(1)
        if cur: blk.append(cur)
        for b in blk:
            nums = re.findall(r"(\d+)", b["loc"])
            if len(nums) >= 2:
                feats.append((b["label"] or b["type"], b["type"], int(nums[0]), int(nums[-1])))
    else:
        xml = None
        for t, b in _sg_blocks(path):
            if t == 10: xml = b.decode("utf-8", "replace")
        if xml:
            for f in re.split(r"(?=<Feature\b)", xml):
                if not f.startswith("<Feature"): continue
                nm = re.search(r'name="([^"]*)"', f); ty = re.search(r'type="([^"]*)"', f)
                segs = re.findall(r'range="(\d+)-(\d+)"', f)
                if nm and segs:
                    feats.append((nm.group(1), ty.group(1) if ty else "-",
                                  min(int(a) for a,_ in segs), max(int(b) for _,b in segs)))
    return feats

def common_prefix(a, b):
    i = 0
    while i < min(len(a), len(b)) and a[i] == b[i]: i += 1
    return i

def common_suffix(a, b):
    i = 0
    while i < min(len(a), len(b)) and a[-1-i] == b[-1-i]: i += 1
    return i

print("配列・feature 読み取り関数を定義しました")

## セル6: 各株の WT / 変異体ファイルを特定

フォルダ名の規則（`<株>_eptA_WT_site` / `<株>_eptA_mutant_site` の下の `Sequence/`）から
自動で拾う。見つからなければ `EPTA_FILES` に手で書く。

In [ ]:
# フォルダ名の接頭辞 -> 株ID
FOLDER2SID = {
    "BthetaVPI":             "Bth_VPI",
    "BthetaCE91_st08":       "Bth_St8",
    "Buniformis":            "Bun_8492",
    "OdoribacteriaceaCE91_st23": "Odo_St23",
    "OdoribacteriaceaCE91_st24": "Odo_St24",   # 参考として解析はするがリファレンスには入れない
}

EPTA_FILES = {}
for d in sorted(EPTA_DIR.glob("*_eptA_*_site")) if EPTA_DIR.exists() else []:
    pre = d.name.split("_eptA_")[0]
    sid = FOLDER2SID.get(pre)
    if not sid: continue
    kind = "WT" if d.name.endswith("_WT_site") else "MUT"
    seqs = [f for f in (d/"Sequence").glob("*") if f.suffix.lower() in (".gb", ".gbk", ".dna")] \
           if (d/"Sequence").exists() else []
    if seqs: EPTA_FILES.setdefault(sid, {})[kind] = seqs[0]

print(f"{'株ID':<10}{'WT':<6}{'MUT':<6}")
for sid in list(FOLDER2SID.values()):
    e = EPTA_FILES.get(sid, {})
    print(f"{sid:<10}{'OK' if 'WT' in e else '—':<6}{'OK' if 'MUT' in e else '—':<6}")
    for k, v in e.items(): print(f"           {k}: {v.name[:70]}")

## セル7: 欠失・挿入区間を求め、eptA CDS 上の位置に換算

出力の `predicted_log2FC_genotype_only` が重要。**発現制御に変化がない場合に
`eptA_full` で観測されるはずの log2FC** で、ステージ5の陽性対照の基準線になる。

Mac で実行した結果（参考値）:

| 株 | eptA CDS | 変異 | CDS内の位置 | CDS残存 | 予測log2FC |
|---|---|---|---|---|---|
| Bth_VPI | 1..1506 | 欠失 503 bp | 873..1375 | 1003/1506 | **−0.59** |
| Bth_St8 | 1184..2887 | 欠失 1689 bp | 16..1704 | 15/1704 | **−6.83** |
| Bun_8492 | 1571..3271 | 欠失 575 bp | 200..774 | 1126/1701 | **−0.60** |
| Odo_St23 | 1..1761 | 挿入 1443 bp | CDS 337位の直後 | 分断 | 0 〜 −2.39 |
| Odo_St24 | 1..1761 | 挿入 1445 bp | CDS 347位の直後 | 分断 | 0 〜 −2.34 |

In [ ]:
import math

# 株 -> (eptA CDS の断片上の start, end, strand, ゲノム上の断片開始, contig)
#   ゲノム座標がファイル名から分かるものは入れてある。None は §C で minimap2 で求める
EPTA_CDS = {
    "Bth_VPI":  (1,    1506, "+", None,    None),
    "Bth_St8":  (1184, 2887, "+", 2841992, "BQNN01000001.1"),
    "Bun_8492": (1571, 3271, "+", None,    None),
    "Odo_St23": (1,    1761, "-", 5080080, "BQOE01000001.1"),
    "Odo_St24": (1,    1761, "+", 330286,  "BQOG01000001.1"),
}

def analyse_eptA(sid):
    e = EPTA_FILES.get(sid, {})
    if "WT" not in e or "MUT" not in e: return None
    w, m = load_seq(e["WT"]), load_seq(e["MUT"])
    pre, suf = common_prefix(w, m), common_suffix(w, m)
    s0, e0 = pre + 1, len(w) - suf            # WT断片上の変異区間（1-based inclusive）
    dele, ins = len(w) - pre - suf, len(m) - pre - suf
    c0, c1, strand, gstart, contig = EPTA_CDS[sid]
    L = c1 - c0 + 1
    if dele > 0:
        ov  = max(0, min(e0, c1) - max(s0, c0) + 1)
        ret = L - ov
        p0 = (s0 - c0 + 1) if strand == "+" else (c1 - e0 + 1)
        p1 = (e0 - c0 + 1) if strand == "+" else (c1 - s0 + 1)
        cdspos = f"{p0}..{p1}"
        pred = f"{math.log2(ret/L):+.2f}" if ret else "≈-inf"
        ipos = None
    else:
        ov, ret = 0, L
        ipos = (s0 - c0) if strand == "+" else (c1 - s0 + 1)   # CDS内の挿入点
        cdspos = f"CDS {ipos} 位の直後"
        pred = f"0 〜 {math.log2(ipos/L):+.2f}"
    return dict(strain=sid, contig=contig or "", genome_frag_start=gstart or "", strand=strand,
                frag_len=len(w), cds_frag_start=c0, cds_frag_end=c1, cds_len=L,
                mut_type="deletion" if dele else "insertion",
                mut_frag_start=s0, mut_frag_end=e0, del_bp=dele, ins_bp=ins,
                insert_pos_in_cds=ipos or "", cds_pos_of_mutation=cdspos,
                cds_deleted_bp=ov, cds_retained_bp=ret,
                predicted_log2FC_genotype_only=pred,
                genome_mut_start=(gstart + s0 - 1) if gstart else "",
                genome_mut_end=(gstart + e0 - 1) if gstart else "")

eptA_rows = [r for r in (analyse_eptA(s) for s in FOLDER2SID.values()) if r]
df_eptA = pd.DataFrame(eptA_rows)
show(df_eptA[["strain","cds_frag_start","cds_frag_end","cds_len","mut_type","del_bp","ins_bp",
              "cds_pos_of_mutation","cds_retained_bp","predicted_log2FC_genotype_only"]],
     "eptA 変異の構造")

spec = QC_DIR/"eptA_mutation_spec.tsv"
df_eptA.to_csv(spec, sep="\t", index=False)
print("\n保存:", spec)

## セル8: WT断片をFASTAに書き出す（§C でゲノムに当てる）

In [ ]:
frag_fa = REF_DIR/"eptA_WT_fragments.fna"
recs = []
for sid in FOLDER2SID.values():
    e = EPTA_FILES.get(sid, {})
    if "WT" in e: recs.append((f"{sid}_WTfragment", load_seq(e["WT"])))
write_fasta(frag_fa, recs)
print(f"{len(recs)} 断片 → {frag_fa}")
for n, s in recs: print(f"  {n:<24}{len(s):>6} bp")

---
# C. eptA 断片をゲノムにマップして座標を確定

**`Bth_VPI` と `Bun_8492` はファイル名にゲノム座標が無い**ので、断片をゲノムに当てて求める。
他の3株は検算になる（ファイル名の座標と一致するはず）。

## セル9: minimap2 で断片をゲノムに当てる

In [ ]:
if not have("minimap2"):
    print("!! minimap2 がありません")
else:
    combined = REF_DIR/"_tmp_all_genomes.fna"
    write_fasta(combined, [(f"{sid}~{n.split()[0]}", s)
                           for sid in STRAINS for n, s in read_fasta(genomes[sid])])
    paf = QC_DIR/"eptA_fragments_vs_genomes.paf"
    sh(f"{tool_path('minimap2')} -x asm5 -t {THREADS} --secondary=no {combined} {frag_fa} > {paf}",
       log=LOG_DIR/"minimap2_eptA.log")
    best = {}
    for line in open(paf):
        p = line.split("\t")
        q, qlen, qs, qe, strand, tgt, tstart, tend = p[0], int(p[1]), int(p[2]), int(p[3]), p[4], p[5], int(p[7]), int(p[8])
        nmatch, blen = int(p[9]), int(p[10])
        sid = q.replace("_WTfragment", "")
        key = sid
        score = nmatch
        if key not in best or score > best[key]["nmatch"]:
            best[key] = dict(strain=sid, q_len=qlen, q_start=qs, q_end=qe, strand=strand,
                             target=tgt, t_start=tstart, t_end=tend,
                             nmatch=nmatch, blen=blen, ident=round(100*nmatch/blen, 2),
                             q_cov=round(100*(qe-qs)/qlen, 1))
    df_map = pd.DataFrame(best.values())
    show(df_map, "断片 → ゲノムのマッピング（最良ヒット）")
    print()
    print("※ q_cov が 100 に近く ident が 99 以上なら信頼できる。")
    print("   target の株IDが断片の株IDと一致していることも確認すること。")

## セル10: ゲノム座標を確定して spec を更新

断片の `t_start` を使って、CDS と変異区間をゲノム座標に変換する。

**注意: PAF の target 座標は 0-based half-open。** 1-based inclusive に直す。
断片が逆向き（`strand == "-"`）に当たった場合は座標を反転させる。

In [ ]:
if "df_map" in dir():
    upd = []
    for r in eptA_rows:
        sid = r["strain"]
        hit = best.get(sid)
        if not hit or hit["q_cov"] < 95:
            upd.append(r); continue
        tcontig = hit["target"].split("~", 1)[1]
        # 断片の先頭(1-based 1)がゲノムのどこに当たるか
        if hit["strand"] == "+":
            def to_g(p): return hit["t_start"] + (p - 1) + 1        # 1-based
        else:
            def to_g(p): return hit["t_end"] - (p - 1)              # 反転
        r2 = dict(r)
        r2["contig_mapped"]  = tcontig
        r2["frag_strand_on_genome"] = hit["strand"]
        for k_frag, k_gen in [("cds_frag_start","cds_genome_start"), ("cds_frag_end","cds_genome_end"),
                              ("mut_frag_start","mut_genome_start"), ("mut_frag_end","mut_genome_end")]:
            r2[k_gen] = to_g(r[k_frag])
        # 反転した場合 start > end になるので入れ替える
        for a, b in [("cds_genome_start","cds_genome_end"), ("mut_genome_start","mut_genome_end")]:
            if r2[a] > r2[b]: r2[a], r2[b] = r2[b], r2[a]
        upd.append(r2)
    df_eptA = pd.DataFrame(upd)
    cols = ["strain","contig_mapped","frag_strand_on_genome","cds_genome_start","cds_genome_end",
            "mut_type","mut_genome_start","mut_genome_end","predicted_log2FC_genotype_only"]
    show(df_eptA[[c for c in cols if c in df_eptA.columns]], "ゲノム座標（確定）")
    df_eptA.to_csv(spec, sep="\t", index=False)
    print("\n更新:", spec)
    print()
    print("検算: ファイル名から分かっていた株と一致するか")
    for r in upd:
        if r.get("genome_mut_start") and r.get("mut_genome_start"):
            d = abs(int(r["genome_mut_start"]) - int(r["mut_genome_start"]))
            print(f"  {r['strain']:<10} ファイル名 {int(r['genome_mut_start']):>9,} / "
                  f"マップ {int(r['mut_genome_start']):>9,}  差 {d}")

---
# D. ゲノム品質の確認（CheckM2）

完全性と混入を見る。**株間のクロスコンタミがあるとすべてが狂う**ので、
アノテーションの前に一度見ておく。

## セル11: CheckM2

In [ ]:
cm_out = QC_DIR/"checkm2"
if not have("checkm2"):
    print("!! checkm2 がありません")
elif (cm_out/"quality_report.tsv").exists():
    print("既に結果があります")
else:
    tmpdir = QC_DIR/"_cm2_input"; tmpdir.mkdir(exist_ok=True)
    for sid in STRAINS:
        dst = tmpdir/f"{sid}.fna"
        if not dst.exists(): dst.write_bytes(Path(genomes[sid]).read_bytes())
    sh(f"{tool_path('checkm2')} predict --threads {THREADS} --force "
       f"--input {tmpdir} --output-directory {cm_out} -x fna",
       log=LOG_DIR/"checkm2.log", timeout=7200)

if (cm_out/"quality_report.tsv").exists():
    q = pd.read_csv(cm_out/"quality_report.tsv", sep="\t")
    show(q[["Name","Completeness","Contamination","Genome_Size","GC_Content","Total_Coding_Sequences"]],
         "CheckM2")
    print()
    print("目安: Completeness > 95%、Contamination < 5%。")
    print("      Contamination が高い株は株間クロスコンタミを疑う")

---
# E. アノテーション — 既存 locus_tag を使う

## 方針

**NCBI/DDBJ が各ゲノムに付けている既存アノテーションを使う。Bakta は欠けている株だけ。**

|- | 既存アノテーション | Bakta de novo |
|---|---|---|
| locus_tag | `BT_RS13740` — **文献で引ける** | `Bth_VPI_00135` — どこにも無い |
| *B. theta* の PUL 文献との照合 | **できる** | 手作業 |
| rRNA / tRNA / ncRNA | **あり**（PGAP・DFAST とも barrnap + tRNAscan-SE） | あり |
| DB | **不要** | 2〜70 GB |
| 株間での gene caller | RefSeq と DDBJ で揃わない | 揃う |

主解析は**株内**比較なので、株間で gene caller が違っても実害がない。
影響するのは「株あたりの遺伝子数」の記述と ORA の universe の縁だけ。
一方 locus_tag の安定性は、解釈と論文化の段階でずっと効く。

## ★ rRNA / tRNA は必ず GTF に入れる

細菌 metatranscriptome では **rRNA 除去後でも全リードの 10〜50% が rRNA**。
GTF に rRNA feature が無いと:

1. 除去効率をサンプルごとに報告できない（QCの中核）
2. rRNA リードが**隣接CDSに漏れて**偽の発現になる
3. ライブラリサイズの分母が rRNA 量に引きずられ size factor が歪む

**rRNA/tRNA は「数えるが DE 解析からは外す」**。消すのではなく `biotype` で区別する。

## 必要なファイル（株ごと）

| ファイル | 用途 |
|---|---|
| `*_genomic.gff` | CDS・rRNA・tRNA の座標と locus_tag |
| `*_translated_cds.faa` | eggNOG の入力 |

**★ `_protein.faa` ではなく `_translated_cds.faa` を使うこと。**
`_protein.faa` はゲノム内で同一配列のタンパク質を**1レコードに統合**するため、
パラログを取りこぼす。`_translated_cds.faa` は CDS ごとに1レコードで、
ヘッダに `[locus_tag=...]` が入っている。

## セル12: 既存アノテーションの有無を確認

`ANNOT_DIR` に accession ごとのフォルダを置く想定。無ければ NCBI datasets で取得する
（セル13にコマンドあり）。

In [ ]:
# 株 -> NCBI/DDBJ アセンブリ accession（★要確認。分かるものだけ埋める）
ACCESSION = {
    "Bth_VPI":  "GCF_000011065.1",   # B. theta VPI-5482 (RefSeq, NC_004663.1)
    "Bth_St8":  None,                # DDBJ BQNN01 — アセンブリ accession を確認
    "Bun_8492": None,                # B. uniformis ATCC 8492
    "Odo_St23": None,                # DDBJ BQOE01
}

ANNOT_DIR = ROOT/"annotation"; ANNOT_DIR.mkdir(exist_ok=True)

def find_annot(sid):
    """株ごとに gff と translated_cds.faa を探す"""
    d = ANNOT_DIR/sid
    g = next(iter(sorted(d.glob("*.gff*"))), None) if d.exists() else None
    f = next(iter(sorted(d.glob("*translated_cds*"))), None) if d.exists() else None
    if f is None and d.exists():                      # 代替: cds_from_genomic
        f = next(iter(sorted(d.glob("*cds_from_genomic*"))), None)
    return g, f

rows = []
for sid in STRAINS:
    g, f = find_annot(sid)
    n_cds = n_rrna = n_trna = 0
    if g:
        for line in (gzip.open(g, "rt") if str(g).endswith(".gz") else open(g)):
            if line.startswith("#"): continue
            ft = line.split("	")[2] if line.count("	") >= 8 else ""
            if ft == "CDS": n_cds += 1
            elif ft == "rRNA": n_rrna += 1
            elif ft == "tRNA": n_trna += 1
    rows.append({"株ID": sid, "accession": ACCESSION.get(sid) or "?",
                 "GFF": g.name if g else "—", "CDS": n_cds, "rRNA": n_rrna, "tRNA": n_trna,
                 "translated_cds": "OK" if f else "—",
                 "判定": "既存を使う" if (g and n_cds and n_rrna) else "Bakta が必要"})
show(pd.DataFrame(rows), "既存アノテーションの状況")

print()
print("判定の基準: CDS があり、かつ rRNA が 1件以上あること。")
print("  rRNA が 0 なら、そのGFFは CDS のみの簡易版。Bakta に回すか barrnap を別途かける")

## セル13: 足りないものを NCBI datasets で取得

accession が分かっていれば1コマンド。DDBJ の CE91 株は、DDBJ/ENA から
`.gbff` を落として GFF に変換する方が早い場合もある。

In [ ]:
need = [sid for sid in STRAINS if find_annot(sid)[0] is None]
print("アノテーションが無い株:", need or "なし")

for sid in need:
    acc = ACCESSION.get(sid)
    if not acc:
        print(f"  {sid}: accession 不明。NCBI/DDBJ で検索して ACCESSION に記入してください")
        continue
    out = ANNOT_DIR/sid; out.mkdir(exist_ok=True)
    print(f"\n# {sid} ({acc}) — ターミナルで:")
    print(f"cd {out}")
    print(f"datasets download genome accession {acc} \\")
    print(f"  --include gff3,cds,protein,genome --filename {acc}.zip")
    print(f"unzip -o {acc}.zip && find ncbi_dataset -type f -exec mv {{}} . \\;")

print()
print("# datasets が無ければ FTP から直接:")
print("#   https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/011/065/GCF_000011065.1_ASM1106v1/")
print("# DDBJ の BQNN01 等は https://www.ncbi.nlm.nih.gov/nuccore/BQNN01000001 から")
print("#   'Send to > File > GenBank (full)' でも取れる")

## セル14: 既存アノテーションが取れない株だけ Bakta

**ここまでで全株揃っていればこのセルは何もしない。**
Bakta を使う場合、DB は **light（約2GB）で十分**（CDS予測・rRNA/tRNA同定は full と同じ。
違うのは UniRef によるタンパク質名の精度だけで、機能アノテーションは §F の eggNOG が担当する）。

In [ ]:
need = [sid for sid in STRAINS if find_annot(sid)[0] is None]
if not need:
    print("全株に既存アノテーションがあります。Bakta は不要です。")
elif not have("bakta"):
    print("!! bakta がありません。必要な株:", need)
elif not BAKTA_DB.exists():
    print(f"!! Bakta DB がありません: {BAKTA_DB}")
    print("# light で十分（約2GB）:")
    print(f"{tool_path('bakta_db') or 'bakta_db'} download --output {BAKTA_DB.parent} --type light")
else:
    for sid in need:
        out = BAKTA_DIR/sid
        if (out/f"{sid}.gff3").exists(): print(f"[スキップ] {sid}"); continue
        out.mkdir(parents=True, exist_ok=True)
        sh(f"{tool_path('bakta')} --db {BAKTA_DB} --output {out} --prefix {sid} "
           f"--locus-tag {sid} --threads {THREADS} --force {genomes[sid]}",
           log=LOG_DIR/f"bakta_{sid}.log", timeout=10800)

---
# F. eggNOG-mapper — 機能アノテーション

KO / COG / CAZy / PFAM を既存の locus_tag に紐づける。
**ステージ6の機能解析はここの出力に依存する。**

## セル15: 入力の faa を揃える

`_translated_cds.faa` のヘッダから locus_tag を取り出して、**ヘッダを locus_tag に付け替える**。
こうすると eggNOG の `query` 列がそのまま locus_tag になり、後の結合が単純になる。

In [ ]:
faa_dir = EMAP_DIR/"input"; faa_dir.mkdir(parents=True, exist_ok=True)
faa_for = {}

for sid in STRAINS:
    gff, faa = find_annot(sid)
    dst = faa_dir/f"{sid}.faa"
    if faa:
        recs, seen = [], set()
        for name, seq in read_fasta(faa):
            m = re.search(r"\[locus_tag=([^\]]+)\]", name)
            lt = m.group(1) if m else name.split()[0].split("|")[-1]
            if lt in seen: continue              # 同一 locus_tag の重複を除く
            seen.add(lt); recs.append((lt, seq.rstrip("*")))
        write_fasta(dst, recs); faa_for[sid] = dst
        print(f"{sid:<10} {len(recs):>5} タンパク質  ({faa.name})")
    else:
        bfaa = BAKTA_DIR/sid/f"{sid}.faa"
        if bfaa.exists():
            dst.write_bytes(bfaa.read_bytes()); faa_for[sid] = dst
            print(f"{sid:<10} Bakta の .faa を使用")
        else:
            print(f"{sid:<10} !! faa がありません")

## セル16: eggNOG-mapper を実行
1株あたり数十分。

In [ ]:
if not have("emapper.py"):
    print("!! emapper.py がありません")
elif not EGGNOG_DB.exists():
    print(f"!! eggNOG DB がありません: {EGGNOG_DB}")
    print(f"download_eggnog_data.py --data_dir {EGGNOG_DB} -y")
else:
    for sid, faa in faa_for.items():
        out = EMAP_DIR/sid
        if (EMAP_DIR/f"{sid}.emapper.annotations").exists():
            print(f"[スキップ] {sid}"); continue
        sh(f"{tool_path('emapper.py')} -i {faa} --itype proteins -o {out} "
           f"--data_dir {EGGNOG_DB} --cpu {THREADS} --override",
           log=LOG_DIR/f"emapper_{sid}.log", timeout=21600)

print()
for sid in STRAINS:
    a = EMAP_DIR/f"{sid}.emapper.annotations"
    if a.exists():
        n = sum(1 for l in open(a) if not l.startswith("#"))
        print(f"  {sid:<10} アノテーション {n:>5} 件")

---
# G. 統合リファレンスの構築

## 守ること

1. **配列を連結して1本にしない。** ファイルを連結した multi-FASTA にし、contig は独立レコードのまま
2. **contig名の先頭に株IDを付ける**（`株ID~contig名`）。Bakta は contig を `contig_1` 等に
   改名するため、そのままでは株間で名前が衝突する
3. **マウスゲノムを decoy として入れる。** 宿主RNAの誤マップを防ぐ
4. **pME008 を入れる。** 両群に存在するので、無いと両群とも unmapped になる

## セル15: 統合FASTA を作る

In [ ]:
ref_fna = REF_DIR/"oligo_eptA.fna"
counts = {}

with open(ref_fna, "w") as out:
    def emit(prefix, path, is_gz=False):
        n = 0
        for name, seq in read_fasta(path):
            cid = name.split()[0]
            out.write(f">{prefix}~{cid}\n")
            for i in range(0, len(seq), 60): out.write(seq[i:i+60]+"\n")
            n += 1
        counts[prefix] = n

    for sid in STRAINS:
        emit(sid, genomes[sid])

    # pME008（GenBank から配列を取り出す）
    if PME008_GB and Path(PME008_GB).exists():
        s = _gb_seq(PME008_GB)
        out.write(">pME008~pSL5053\n")
        for i in range(0, len(s), 60): out.write(s[i:i+60]+"\n")
        counts["pME008"] = 1
        print(f"pME008: {len(s):,} bp")

    # マウス decoy
    if MOUSE_FNA.exists():
        emit("mouse", MOUSE_FNA)
    else:
        print("!! マウスゲノムが無いので decoy なしで作成しました（後で入れ直すこと）")

print()
for k, v in counts.items(): print(f"  {k:<12}{v:>6} contig")
print(f"\n{ref_fna}  ({ref_fna.stat().st_size/1e9:.2f} GB)")

## セル16: contig名の検算
株IDが全てに付いているか、重複が無いか。

In [ ]:
names = [n for n, _ in read_fasta(ref_fna)]
print(f"contig 総数 {len(names):,} / ユニーク {len(set(names)):,}")
assert len(names) == len(set(names)), "!! contig名が重複しています"
c = Counter(n.split("~")[0] for n in names)
for k, v in c.most_common(): print(f"  {k:<12}{v:>7}")
print("\n先頭5件:", *["  >"+n for n in names[:5]], sep="\n")

---
# H. 統合GTF の構築

**このnotebookで最も重要な成果物。** 以下を属性として持たせる。

| 属性 | 由来 | 用途 |
|---|---|---|
| `gene_id` | Bakta の locus_tag | カウントの単位 |
| `strain` | contig名の接頭辞 | 株ごとの集計 |
| `feature_class` | Bakta の source | **CDS だけに絞るため**（`Prodigal` = CDS） |
| `Knumber` / `COG` / `CAZy` / `PFAM` | eggNOG | ステージ6の機能解析 |
| `product` | Bakta | 解釈 |
| `mappability` | §J で後から追加 | 株レベルの結論を出すときのフィルタ |

さらに **eptA の3分割 feature** を別行として足す（§4 の方針）。

```
eptA_full       元のCDS全長        → DE解析から除外する
eptA_retained   両群に共通する領域   → ★ここで比較する
eptA_deleted    K群で欠失する領域    → 遺伝子型の確認用
```

Odoribacter は挿入なので `eptA_5p` / `eptA_3p` に分ける。

## GFF + eggNOG を1つの表にまとめる

**NCBI/DDBJ GFF と Bakta GFF3 の両方に対応する。** どちらも `locus_tag=` を持つので、
そこを `gene_id` にすれば統一できる。

`biotype` 列を立てて **CDS / rRNA / tRNA / ncRNA を区別する**。rRNA は数えるが
DE 解析からは外すので、消さずに残す。

In [ ]:
def parse_attrs(s):
    d = {}
    for kv in s.rstrip(";").split(";"):
        if "=" in kv:
            k, v = kv.split("=", 1)
            d[k] = v.replace("%2C", ",").replace("%3B", ";").replace("%20", " ")
    return d

KEEP = {"CDS":"CDS", "rRNA":"rRNA", "tRNA":"tRNA", "tmRNA":"ncRNA",
        "ncRNA":"ncRNA", "RNase_P_RNA":"ncRNA", "SRP_RNA":"ncRNA", "antisense_RNA":"ncRNA"}

rows = []
for sid in STRAINS:
    gff, _ = find_annot(sid)
    src_label = "NCBI"
    if gff is None:
        gff, src_label = BAKTA_DIR/sid/f"{sid}.gff3", "Bakta"
    if not Path(gff).exists(): print(f"!! {sid}: GFF なし"); continue
    op = gzip.open if str(gff).endswith(".gz") else open
    for line in op(gff, "rt"):
        if line.startswith("#"): continue
        p_ = line.rstrip("\n").split("\t")
        if len(p_) < 9 or p_[2] not in KEEP: continue
        a = parse_attrs(p_[8])
        lt = a.get("locus_tag") or a.get("ID", "").replace("cds-", "").replace("gene-", "")
        if not lt: continue
        rows.append(dict(strain=sid, contig=f"{sid}~{p_[0]}", annot_source=src_label,
                         biotype=KEEP[p_[2]], start=int(p_[3]), end=int(p_[4]), strand=p_[6],
                         gene_id=lt, gene_name=a.get("gene", ""),
                         product=a.get("product", ""), protein_id=a.get("protein_id", "")))

gtf_df = pd.DataFrame(rows).drop_duplicates(subset=["strain","gene_id","biotype","start","end"])
print("feature:", len(gtf_df))
show(gtf_df.groupby(["strain","annot_source","biotype"]).size().unstack(fill_value=0),
     "株 × biotype")
print()
print("★ rRNA が各株に 3〜30 程度あることを確認。0 の株は rRNA 注釈が欠けている")

## eggNOG を結合

`query` 列が locus_tag になっているので、`gene_id` でそのまま結合できる。

In [ ]:
emap_all = []
for sid in STRAINS:
    a = EMAP_DIR/f"{sid}.emapper.annotations"
    if not a.exists(): continue
    e = pd.read_csv(a, sep="	", comment="#", header=None, low_memory=False)
    cols = ["query","seed_ortholog","evalue","score","eggNOG_OGs","max_annot_lvl","COG_category",
            "Description","Preferred_name","GOs","EC","KEGG_ko","KEGG_Pathway","KEGG_Module",
            "KEGG_Reaction","KEGG_rclass","BRITE","KEGG_TC","CAZy","BiGG_Reaction","PFAMs"]
    e.columns = cols[:e.shape[1]]
    e["strain"] = sid
    emap_all.append(e)

if emap_all:
    emap = pd.concat(emap_all, ignore_index=True)
    use = [c for c in ["query","KEGG_ko","KEGG_Pathway","KEGG_Module","COG_category",
                       "CAZy","PFAMs","EC","Preferred_name"] if c in emap.columns]
    gtf_df = gtf_df.merge(emap[use].drop_duplicates(subset=["query"]),
                          left_on="gene_id", right_on="query", how="left").drop(columns=["query"])
    cds = gtf_df[gtf_df.biotype == "CDS"]
    print(f"CDS {len(cds):,} 件中 KO 付与 {cds.KEGG_ko.notna().sum():,} "
          f"({100*cds.KEGG_ko.notna().mean():.1f}%)")
    print()
    print("※ KO 付与率は細菌なら 50〜70% が普通。極端に低ければ faa のヘッダを疑う")
else:
    print("!! eggNOG の結果がありません")

## セル18: eptA の3分割 feature を作る

`eptA_mutation_spec.tsv` のゲノム座標から、GTF に足す行を組み立てる。

**対応する Bakta の CDS を座標で突き止めてから分割する。** Bakta が付けた locus_tag を
`parent_gene_id` として記録し、元の CDS 行には `exclude_from_DE=true` を立てる。

In [ ]:
spec_df = pd.read_csv(spec, sep="\t")
extra_rows, exclude_ids = [], set()

for _, r in spec_df.iterrows():
    sid = r["strain"]
    if sid not in STRAINS: continue
    gs, ge = r.get("cds_genome_start"), r.get("cds_genome_end")
    ms, me = r.get("mut_genome_start"), r.get("mut_genome_end")
    if pd.isna(gs) or pd.isna(ms):
        print(f"!! {sid}: ゲノム座標が未確定（§C を実行してください）"); continue
    gs, ge, ms, me = int(gs), int(ge), int(ms), int(me)

    # 座標が重なる Bakta CDS を探す
    cand = gtf_df[(gtf_df.strain == sid) & (gtf_df.biotype == "CDS") &
                  (gtf_df.start <= ge) & (gtf_df.end >= gs)]
    if cand.empty:
        print(f"!! {sid}: 対応する Bakta CDS が見つかりません ({gs:,}-{ge:,})"); continue
    cds = cand.iloc[(cand.start - gs).abs().argsort()].iloc[0]
    parent = cds.gene_id; contig = cds.contig; strand = cds.strand
    exclude_ids.add(parent)
    print(f"{sid:<10} eptA = {parent}  {contig}:{cds.start:,}-{cds.end:,} ({strand})")

    def add(suffix, s, e, note):
        if e < s: return
        extra_rows.append(dict(strain=sid, contig=contig, annot_source="eptA_spec", biotype="CDS",
                               start=s, end=e, strand=strand,
                               gene_id=f"{parent}_{suffix}", gene_name=f"eptA_{suffix}",
                               product=note, parent_gene_id=parent))

    if r["mut_type"] == "deletion":
        add("deleted", ms, me, "eptA: K群で欠失する領域（遺伝子型の確認用）")
        if ms > cds.start: add("retained5p", cds.start, ms-1, "eptA: 両群に共通（5'側）")
        if me < cds.end:   add("retained3p", me+1, cds.end, "eptA: 両群に共通（3'側）")
    else:
        # 挿入: 挿入点で5'/3'に分ける（strand を考慮）
        ip = ms
        if strand == "+":
            add("5p", cds.start, ip, "eptA: 挿入点より5'側（両群で転写される）")
            add("3p", ip+1, cds.end, "eptA: 挿入点より3'側（K群で落ちれば転写終結）")
        else:
            add("5p", ip, cds.end, "eptA: 挿入点より5'側（両群で転写される）")
            add("3p", cds.start, ip-1, "eptA: 挿入点より3'側（K群で落ちれば転写終結）")

df_extra = pd.DataFrame(extra_rows)
if len(df_extra):
    show(df_extra[["strain","gene_id","start","end","product"]], "追加する eptA feature")
print("\nDE解析から除外する元CDS:", exclude_ids)

## セル19: pME008 の注釈を GTF 行に変換
Baktaは使わず、pSL5053 の既存アノテーションを活かす。

In [ ]:
pme_rows = []
if PME008_GB and Path(PME008_GB).exists():
    for name, typ, s, e in load_features(PME008_GB):
        if typ not in ("CDS", "gene", "misc_feature", "promoter"): continue
        if e - s < 150 and typ != "promoter": continue        # 短いprimer等は落とす
        gid = "pME008_" + re.sub(r"[^A-Za-z0-9]+", "_", name)[:40].strip("_")
        pme_rows.append(dict(strain="pME008", contig="pME008~pSL5053", annot_source="pME008_gb",
                             biotype="CDS", start=s, end=e, strand="+",
                             gene_id=gid, gene_name=name[:40], product=f"pME008: {name}"))
df_pme = pd.DataFrame(pme_rows).drop_duplicates(subset=["gene_id"])
if len(df_pme):
    show(df_pme[["gene_id","start","end","product"]].head(30), f"pME008 feature（{len(df_pme)}件）")
    print()
    print("※ tetX1 / Cas / Tns / PcfxA が入っているか確認すること。")
    print("   重複する label があれば gene_id が衝突しないよう drop_duplicates 済み")

## セル20: GTF を書き出す

In [ ]:
gtf_path = REF_DIR/"oligo_eptA.gtf"
all_df = pd.concat([gtf_df.assign(parent_gene_id=""), df_extra, df_pme.assign(parent_gene_id="")],
                   ignore_index=True)
all_df["exclude_from_DE"] = all_df.gene_id.isin(exclude_ids)

def esc(v):
    return str(v).replace('"', "'").replace(";", ",") if pd.notna(v) else "NA"

ATTRS = ["gene_name","product","strain","biotype","annot_source","protein_id","parent_gene_id",
         "KEGG_ko","KEGG_Pathway","KEGG_Module","COG_category","CAZy","PFAMs","EC","exclude_from_DE"]

with open(gtf_path, "w") as f:
    for _, r in all_df.iterrows():
        a = f'gene_id "{r.gene_id}"; transcript_id "{r.gene_id}"; '
        a += "".join(f'{k} "{esc(r.get(k))}"; ' for k in ATTRS if k in all_df.columns)
        f.write(f"{r.contig}\t{r.annot_source}\ttranscript\t{r.start}\t{r.end}\t.\t{r.strand}\t0\t{a.strip()}\n")

print(f"{len(all_df):,} 行 → {gtf_path}")
print()
show(all_df.groupby(["strain","biotype"]).size().unstack(fill_value=0), "株 × biotype")
print()
print("CDS（Prodigal）だけに絞ったときの数:")
print("  ", (all_df.biotype == "CDS").sum())

---
# I. index の構築

bowtie2 と salmon の両方を作る。**Bth 2株が近縁なので Salmon を並走させる**
（featureCounts の fraction mode は曖昧リードを 1/NH で等分するだけだが、
Salmon の EM は一意リードの情報を使って確率的に再配分する）。

## セル21: bowtie2 index

In [ ]:
bt2 = REF_DIR/"bt2"/"oligo_eptA"
bt2.parent.mkdir(exist_ok=True)
if Path(str(bt2)+".1.bt2").exists() or Path(str(bt2)+".1.bt2l").exists():
    print("既にあります:", bt2)
elif have("bowtie2-build"):
    sh(f"{tool_path('bowtie2-build')} --threads {THREADS} {ref_fna} {bt2}",
       log=LOG_DIR/"bt2build.log", timeout=10800)

## セル22: salmon index（decoy-aware）

CDS の配列を転写産物として、**ゲノム全体を decoy にする**。
マウスゲノムも decoy に入るので、宿主RNAがCDSに吸い寄せられない。

In [ ]:
salmon_dir = REF_DIR/"salmon"
if (salmon_dir/"info.json").exists():
    print("既にあります:", salmon_dir)
elif have("salmon"):
    # CDS だけを転写産物FASTAにする
    cds_fa = REF_DIR/"oligo_eptA_cds.fna"
    seqs = {n.split()[0]: s for n, s in read_fasta(ref_fna)}
    recs = []
    for _, r in all_df[all_df.biotype.isin(["CDS"])].iterrows():
        s = seqs.get(r.contig)
        if not s: continue
        sub = s[r.start-1:r.end]
        if r.strand == "-":
            sub = sub[::-1].translate(str.maketrans("ACGTNacgtn","TGCANtgcan"))
        recs.append((r.gene_id, sub))
    write_fasta(cds_fa, recs)
    print(f"転写産物 {len(recs):,} 本 → {cds_fa}")

    decoy = REF_DIR/"decoys.txt"
    decoy.write_text("\n".join(n.split()[0] for n, _ in read_fasta(ref_fna)) + "\n")
    gentrome = REF_DIR/"gentrome.fna"
    sh(f"cat {cds_fa} {ref_fna} > {gentrome}", quiet=True)
    sh(f"{tool_path('salmon')} index -t {gentrome} -d {decoy} -i {salmon_dir} "
       f"-p {THREADS} -k 31", log=LOG_DIR/"salmon_index.log", timeout=10800)

---
# J. mappability の実測

**ゲート1の積み残し。** Bth_VPI ↔ Bth_St8 の共通領域（96.8%）が、
実リード長 150 bp でどこまで株を区別できるかを測る。

ステージ1の見立て: ANI 98% = 2%分岐なので、150 bp リードには平均約3塩基の差が入り、
bowtie2 は区別できるはず。**固有領域の206 CDSに限定する前に、必ず実測する。**

## セル23: 擬似リードを作ってマップし戻す

In [ ]:
STEP = 10
sim_fa = MAP_DIR/"sim_reads.fa"
mapq_tsv = MAP_DIR/"cds_mappability.tsv"

if not mapq_tsv.exists():
    # CDS から READLEN の擬似リードを刻む（リード名に 株~遺伝子ID を埋める）
    seqs = {n.split()[0]: s for n, s in read_fasta(ref_fna)}
    n_read = 0
    with open(sim_fa, "w") as out:
        for _, r in all_df[all_df.biotype == "CDS"].iterrows():
            s = seqs.get(r.contig)
            if not s: continue
            sub = s[r.start-1:r.end]
            for i in range(0, max(1, len(sub)-READLEN+1), STEP):
                out.write(f">{r.strain}~{r.gene_id}~{i}\n{sub[i:i+READLEN]}\n"); n_read += 1
    print(f"擬似リード {n_read:,} 本（{READLEN} bp、{STEP} bp刻み）")

    tot, uniq, conf, src_of = Counter(), Counter(), defaultdict(Counter), {}
    cmd = (f"{tool_path('bowtie2')} -f -x {bt2} -U {sim_fa} -p {THREADS} --no-unal "
           f"2> {LOG_DIR/'bt2_mappability.log'} | {tool_path('samtools')} view -")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, text=True, bufsize=1<<20)
    for line in proc.stdout:
        p = line.split("\t", 6)
        if len(p) < 5: continue
        s_, gid, _ = p[0].split("~", 2)
        dst = p[2].split("~", 1)[0]
        k = f"{s_}~{gid}"; src_of[k] = s_
        tot[k] += 1
        if int(p[4]) >= 10: uniq[k] += 1
        elif dst != s_: conf[k][dst] += 1
    proc.wait()

    with open(mapq_tsv, "w") as out:
        out.write("strain\tgene\tn_sim\tn_unique\tmappability\ttop_confused_with\n")
        for k, t in tot.items():
            b = conf[k].most_common(1)
            out.write(f"{src_of[k]}\t{k}\t{t}\t{uniq[k]}\t{uniq[k]/t:.3f}\t{b[0][0] if b else '-'}\n")
    print(f"{len(tot):,} CDS → {mapq_tsv}")
else:
    print("既に結果があります:", mapq_tsv)

## セル24: 株ごとの mappability

In [ ]:
mp = pd.read_csv(mapq_tsv, sep="\t")
s = (mp.groupby("strain")
       .agg(CDS数=("gene","size"), 平均=("mappability","mean"), 中央値=("mappability","median"),
            区別可能=("mappability", lambda x: (x>=0.5).sum()),
            区別困難=("mappability", lambda x: (x<0.5).sum()),
            ほぼ不能=("mappability", lambda x: (x<0.1).sum()))
       .reset_index())
s["区別可能_pct"] = (100*s["区別可能"]/s["CDS数"]).round(1)
show(s, f"CDSごとの株判別可能性（リード長 {READLEN} bp）")

print()
print("mappability < 0.5 のCDSの混同相手:")
bad = mp[(mp.mappability < 0.5) & (mp.top_confused_with != "-")]
for _, r in (bad.groupby(["strain","top_confused_with"]).size()
               .reset_index(name="CDS数").sort_values("CDS数", ascending=False).iterrows()):
    print(f"  {r.strain:<10} ↔ {r.top_confused_with:<10}{r['CDS数']:>6} CDS")

print()
print("★ ゲート1の積み残しの答え:")
for sid in ("Bth_VPI","Bth_St8"):
    v = mp.loc[mp.strain == sid, "mappability"]
    if len(v):
        print(f"   {sid}: 区別可能なCDS {int((v>=0.5).sum()):,} / {len(v):,} "
              f"({100*(v>=0.5).mean():.1f}%)")
print("   固有領域は206/230 CDS。これを大きく上回れば、共通領域の大半も使える")

## セル25: mappability を GTF に書き戻す

In [ ]:
mp2 = mp.copy()
mp2["gene_id"] = mp2.gene.str.split("~").str[1]
mapd = dict(zip(mp2.gene_id, mp2.mappability))
confd = dict(zip(mp2.gene_id, mp2.top_confused_with))

out_gtf = REF_DIR/"oligo_eptA.gtf"
tmp = REF_DIR/"_tmp.gtf"
n = 0
with open(out_gtf) as fin, open(tmp, "w") as fout:
    for line in fin:
        m = re.search(r'gene_id "([^"]+)"', line)
        gid = m.group(1) if m else None
        if gid in mapd:
            line = line.rstrip("\n") + f' mappability "{mapd[gid]:.3f}"; confused_with "{confd[gid]}";\n'
            n += 1
        fout.write(line)
tmp.replace(out_gtf)
print(f"{n:,} 行に mappability を追加 → {out_gtf}")

---
# K. 成果物の確認

## セル26: できたものを一覧

In [ ]:
want = [("統合FASTA", ref_fna), ("統合GTF", gtf_path),
        ("bowtie2 index", Path(str(bt2)+".1.bt2")), ("salmon index", salmon_dir/"info.json"),
        ("eptA spec", spec), ("mappability", mapq_tsv),
        ("CheckM2", cm_out/"quality_report.tsv")]
print(f"{'成果物':<18}{'状態':<6}{'サイズ':>12}  パス")
for nm, p in want:
    ok = Path(p).exists()
    sz = f"{Path(p).stat().st_size/1e6:,.1f} MB" if ok else "-"
    print(f"{nm:<18}{'OK' if ok else '!!':<6}{sz:>12}  {p}")

print()
print("GTF の中身:")
if gtf_path.exists():
    src = Counter(re.search(r'biotype "([^"]*)"', l).group(1)
                  for l in open(gtf_path) if 'biotype "' in l)
    for k, v in src.most_common(): print(f"  {k:<16}{v:>7}")
    print()
    print("eptA 関連の行:")
    for l in open(gtf_path):
        if "eptA_" in l:
            p = l.split("\t")
            g = re.search(r'gene_id "([^"]+)"', l)
            print(f"  {g.group(1):<28}{p[0]:<24}{p[3]:>10}-{p[4]:<10}")

## 次: ステージ3（サンプル処理）

ゲートではないので、確認できたらそのまま進んでよい。

```
FastQC/MultiQC
  → fastp（アダプター明示: P7 AGATCGGAAGAGCACACGTCTGAACTCCAGTCA /
                            P5 AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGT）
  → Bowtie2 → samtools sort/index
  → featureCounts -s 2 -p --countReadPairs -B --fraction -M
  → Salmon（並走）
```

**`-s 2` を忘れないこと。** 同系統の過去データで rev/fwd = 20〜55:1 を実測済み。

その次のステージ4がゲート2（定着判定と遺伝子型確認）。